# Phase 3 — Fold-0 CNN training (Kaggle T4)

Trains the CNN encoder + cross-attention decoder on fold 0 of the Phase 1/2
tag-grouped CV. Inputs come from `wellbore-prediction-code` (the latest
`src/` bundle) and `wellbore-prediction-data` (the original raw competition
data — `train/` folder of `*__horizontal_well.csv`, `*__typewell.csv`).

Outputs `fold_models/fold_0.pt` and `oof_fold_0.parquet` to `/kaggle/working/`.
Download both and version them as a new dataset
(`wellbore-prediction-nn-checkpoints`) for the submission notebook.

In [ ]:
!pip install -q torch pyarrow

In [ ]:
import os, shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo")
REPO.mkdir(parents=True, exist_ok=True)

# Copy code dataset into the repo root
src_zip = Path("/kaggle/input/wellbore-prediction-code/src.zip")
if src_zip.exists():
    shutil.unpack_archive(str(src_zip), REPO)
else:
    # Fall back to copying the unzipped tree
    shutil.copytree("/kaggle/input/wellbore-prediction-code/src", REPO / "src", dirs_exist_ok=True)

os.chdir(REPO)
print("Repo contents:", list(REPO.iterdir()))

In [ ]:
os.environ["DATA_DIR"] = "/kaggle/input/wellbore-prediction-data"
os.environ["ARTEFACT_DIR"] = "/kaggle/working/artefacts"
os.environ["MODE"] = "fold"
os.environ["MODEL_KIND"] = "cnn"
os.environ["FOLD"] = "0"
os.environ["N_EPOCHS"] = "50"
os.environ["BATCH_SIZE"] = "16"
os.environ["DEVICE"] = "cuda"

# Sanity print
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
!cd /kaggle/working/repo && python -m src.nn.cli

In [ ]:
out = Path("/kaggle/working/artefacts/nn/cnn")
for p in out.rglob("*"):
    print(p, p.stat().st_size if p.is_file() else "")